# Amortized EI with a PFN-VLA action expert: does reading the frozen PFN help?

## Description

**Question.** A policy amortized over the prior has to *read* the PFN's belief. The VLA recipe does this with a small action expert that cross-attends to a frozen backbone's per-layer keys and values (π0). The cleanest test of that architecture is to have it learn a known acquisition function: one forward pass from the context to the distribution of EI's maximizer, with no candidate scoring. **Primary question (A4): does an expert reading `pfn_large`'s context K/V learn amortized EI better than the same expert reading an identically shaped encoder trained from scratch on the same data (D-only)?** If not, the frozen PFN's representation gives the policy nothing that the training data alone doesn't. The predecessor answered no on the small smoke PFN (D-only 0.900 vs 0.889 at d = 6). `pfn_large` is a 6.6 M-parameter model trained 50× longer, so the answer may differ.

**Setup.**
- **States.** BNN-prior tasks at d = 2, split 3 for training (4096 tasks), split 4 for test (256 tasks). From each task:
  - every prefix of an EI trajectory (n_init = 4, T = 20, both run with pfn_large), which is the distribution a BO policy meets;
  - as many random uniform contexts with n ~ U{1, …, 24}.

  That gives 163,840 training and 10,240 test states.
- **Target.** p(x) ∝ exp((EI(x) − max EI) / (0.03 · max EI)) over 4096 shifted-Sobol candidates plus 3 × 64 incumbent perturbations. EI comes from pfn_large. Training uses the 128 heaviest candidates, renormalized. The loss is the forward KL (cross-entropy).
- **Expert.**
  - 8 layers, one per PFN layer, width 128, 4 heads;
  - tokens: an action token (plus the budget left) and the incumbent;
  - head: a 16-component Gaussian mixture in logit space.

  The trunks are:
  - `pfn_all`: layer l reads PFN layer l;
  - `pfn_last`: every layer reads the last PFN layer;
  - `d_only`: reads a PFN with pfn_large's exact architecture, randomly initialized and trained jointly. That is everything the frozen PFN has except its pretraining.
  - `pfn_all` + 256 **probes**: also reads the frozen PFN's final state at 256 fixed Sobol points, i.e. its prediction at query points. This is the predecessor's fix.

  Identical data, steps (20k × 256), optimizer and seed for every trunk.
- **Reference.** EI maximized over 256 random points: a cost-matched alternative to one-pass amortization.

**What to look for.**
- **M1:** the EI ratio EI(x̂)/max EI on test states, for the mixture's mode and for a sample (median and 10th percentile), split by state source (trajectory vs random) and by context size.
- **Closed loop:** 200 fresh tasks (split 5), T = 20. The expert (mode, sample) runs against EI with the full candidate optimizer, compared by the paired geometric-mean regret ratio.

**Decision rule (fixed before the run).**
- **A4 GO** (the frozen representation helps) if the mean paired difference of test-state EI ratios (mode), `pfn_all − d_only`, is ≥ 0.02 with its 95% bootstrap CI above 0.
- **A4 NO-GO** if that CI lies entirely below +0.02, i.e. no advantage worth having.
- **INCONCLUSIVE** otherwise.
- *Secondary (reported, not gating).* `pfn_last` vs `pfn_all` (does depth matter); probes vs `pfn_all`; closed-loop non-inferiority to EI (geometric-mean regret ratio CI upper bound < 1.25).

**Limits.**
- d = 2 only: the predecessor's readout degraded quickly with d, and d = 2 is where a negative result is most informative.
- One seed per trunk. Task-to-task variation is in the CIs; training-seed variation is not. A difference that hinges on it is reported as such.
- In-prior, noise-free.
- The probe trunk reintroduces 256 PFN query tokens per decision, so it is no longer "one pass with no candidate scoring".


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/ei_readout/<sweep_id>"  # set after `just snapshot runs/ei_readout/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.